In [80]:
from typing import override
from torch import Tensor, nn
import torch
import numpy as np


# Network hyperparam
dim = 65
dimT = 50
L = 4
mode = 8
width = 32
lr = 5e-4
du = 2  # 2 for x, y components.
da = du + 2  # 2 for viscosity and pressure

# PDE param
T = (5, 10)
re = 500
alpha = 1
beta = 1
k = 1

# NN


# Fourier Convolution Operator
class FCO(nn.Module):
    def __init__(self, mode: int, width: int) -> None:
        super().__init__()
        self.mode: int = mode
        
        self.R = nn.Parameter(torch.randn(mode, width, width, dtype=torch.complex64))

    @override
    def forward(self, x: Tensor) -> Tensor:
        # x of shape: (B, dim, dim, width)
        shape = x.shape
        x_hat = torch.fft.fftn(x, dim=(1, 2))[:, :self.mode, :self.mode, :]
        x_hat = torch.einsum('mij,bmni->bmnj', self.R, x_hat)
        x_hat = torch.fft.ifftn(x_hat, s=shape)
        x = x_hat.real
        print(x.shape)
        return x


class PINOLayer(nn.Module):
    def __init__(self, mode: int, width: int) -> None:
        super().__init__()

        self.k: nn.Module = FCO(mode, width)
        self.w: nn.Module = nn.Linear(width, width)
        self.sigma: nn.Module = nn.GELU()

    @override
    def forward(self, x: Tensor) -> Tensor:
        w = self.w(x)
        k = self.k(x)
        x = self.sigma(w + k)
        return x


class PINO(nn.Module):
    def __init__(self, da: int, du: int, mode: int, width: int, L: int) -> None:
        super().__init__()

        # uplift
        R = nn.Linear(da, width)
        # projection
        Q = nn.Linear(width, du)

        layers = [PINOLayer(mode, width) for _ in range(L)]
        self.core: nn.Module = nn.Sequential(
            R,
            *layers,
            Q,
        )

    @override
    def forward(self, x: Tensor) -> Tensor:
        return self.core(x)



In [90]:
pino = PINO(da, du, mode, width, L)

In [91]:
Batch = 2
test_input = torch.rand(Batch, dim, dim, da)

In [92]:
output = pino(test_input)

torch.Size([2, 65, 65, 32])
torch.Size([2, 65, 65, 32])
torch.Size([2, 65, 65, 32])
torch.Size([2, 65, 65, 32])


In [77]:
loss = output.sum()


In [78]:
loss.backward()

In [79]:
loss

tensor(706.2964, grad_fn=<SumBackward0>)